# EMS Data Profiling

Checks what the EPCR (Elite) and outcomes (ESO) data can actually answer before any analysis starts.

Run sections 0 through 3 first. They find the tables, flatten them, and pick the columns everything else uses.
If a column is picked wrong, set it by hand in `OVERRIDES` in section 0 and re-run from section 3.

The last section scores each question we want to answer against how complete the fields behind it are.

## 0. Config

In [ ]:
CATALOG = "prod"
EPCR_SCHEMA = "silver_elite_dwgmr"
EPCR_SCHEMA_ALT = "silver_elite_dwamgh"
ESO_TABLE = "prod.silver_frn_qry_eso_ems"

FACT = f"{CATALOG}.{EPCR_SCHEMA}.fact_incident"
DIMS = ["incident", "situation", "disposition", "agency", "patient", "payment", "scene", "cardiacarrest"]

YEAR_MIN = 2023
YEAR_MAX = 2026

OVERRIDES = {
    "incident_date": None,
    "incident_id": None,
    "patient_id": None,
    "payer": None,
    "primary_impression": None,
    "acuity": None,
    "disposition": None,
    "state": None,
    "county": None,
    "zip": None,
    "agency_name": None,
}

BLANKS = ["", "null", "none", "n/a", "na", "unknown", "not recorded", "not applicable",
          "not reporting", "not known", "unable to complete"]

## 1. Helpers

In [ ]:
from pyspark.sql import functions as F

def cols(table):
    try:
        return [f.name for f in spark.table(table).schema.fields]
    except Exception:
        return []

def tables(schema):
    return [r.tableName for r in spark.sql(f"SHOW TABLES IN {CATALOG}.{schema}").collect()]

def search_columns(schema, keys):
    rows = []
    for t in tables(schema):
        for c in cols(f"{CATALOG}.{schema}.{t}"):
            if any(k.lower() in c.lower() for k in keys):
                rows.append((t, c))
    if not rows:
        return spark.createDataFrame([], "table string, column string")
    return spark.createDataFrame(rows, "table string, column string")

def resolve(df, key, keys, avoid=()):
    if OVERRIDES.get(key):
        return OVERRIDES[key]
    for k in keys:
        for c in df.columns:
            if k.lower() in c.lower() and not any(a.lower() in c.lower() for a in avoid):
                return c
    return None

def is_blank(c):
    return F.col(c).isNull() | F.lower(F.trim(F.col(c).cast("string"))).isin(BLANKS)

def completeness(df, columns):
    n = df.count()
    agg = df.agg(*[F.count(F.when(~is_blank(c), 1)).alias(c) for c in columns]).collect()[0].asDict()
    rows = [(k, int(v), n, round(100.0 * v / n, 1) if n else 0.0) for k, v in agg.items()]
    return spark.createDataFrame(rows, "field string, populated long, total long, pct_populated double").orderBy("pct_populated")

def top_values(df, column, n=25):
    total = df.count()
    return (df.groupBy(column).count()
              .withColumn("pct", F.round(100.0 * F.col("count") / total, 2))
              .orderBy(F.desc("count")).limit(n))

def overlap(left, lcol, right, rcol):
    l = left.select(F.col(lcol).cast("string").alias("k")).where("k is not null").distinct()
    r = right.select(F.col(rcol).cast("string").alias("k")).where("k is not null").distinct()
    ln, rn = l.count(), r.count()
    both = l.join(r, "k").count()
    return spark.createDataFrame(
        [(lcol, rcol, ln, rn, both,
          round(100.0 * both / ln, 1) if ln else 0.0,
          round(100.0 * both / rn, 1) if rn else 0.0)],
        "left_col string, right_col string, left_distinct long, right_distinct long, matched long, pct_of_left double, pct_of_right double")

## 2. What tables exist

Also checks whether `silver_elite_dwgmr` / `silver_elite_dwamgh` and the `silver_frn_qry_*` copies hold the same data, which came up as an open question.

In [ ]:
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}").where("databaseName like 'silver%'"))

In [ ]:
inv = []
for s in [EPCR_SCHEMA, EPCR_SCHEMA_ALT]:
    for t in tables(s):
        full = f"{CATALOG}.{s}.{t}"
        try:
            inv.append((s, t, spark.table(full).count(), len(cols(full))))
        except Exception:
            inv.append((s, t, -1, -1))

display(spark.createDataFrame(inv, "schema string, table string, rows long, n_cols long").orderBy(F.desc("rows")))

In [ ]:
display(search_columns(EPCR_SCHEMA, ["payment", "payer", "insur", "medicaid", "billing"]))

In [ ]:
display(search_columns(EPCR_SCHEMA, ["zip", "county", "state", "fips", "census", "lat", "lon"]))

## 3. Flatten the table

Joins `fact_incident` to its dimension tables so one row is one incident with all fields attached. Uses the `Dim_X_FK` to `Dim_X_PK` naming. Dimension columns get the dimension name in front so they do not collide.

In [ ]:
fk_map = {}
for c in cols(FACT):
    if c.lower().endswith("_fk"):
        fk_map[c[:-3].rstrip("_").lower()] = c

print(fk_map)

In [ ]:
import re

odd = []
for d in DIMS + ["incident"]:
    for c in cols(f"{CATALOG}.{EPCR_SCHEMA}.dim_{d}"):
        if not re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", c):
            odd.append((f"dim_{d}", c))
for c in cols(FACT):
    if not re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", c):
        odd.append(("fact_incident", c))

print(len(odd))
for x in odd[:50]:
    print(x)

In [ ]:
import re

def clean(name):
    return re.sub(r"[^0-9a-zA-Z_]", "_", name)

sel = ["fi.*"]
joins = []
for d in DIMS:
    tbl = f"dim_{d}"
    full = f"{CATALOG}.{EPCR_SCHEMA}.{tbl}"
    if tbl not in fk_map or not cols(full):
        print(f"skipped {tbl}")
        continue
    fk = fk_map[tbl]
    pk = fk[:-2] + "PK"
    joins.append(f"LEFT JOIN {full} {d} ON fi.`{fk}` = {d}.`{pk}`")
    for c in cols(full):
        if not c.lower().endswith("_pk"):
            sel.append(f"{d}.`{c}` AS `{d}_{clean(c)}`")

sql = f"SELECT {', '.join(sel)} FROM {FACT} fi " + " ".join(joins)
spark.sql(sql).createOrReplaceTempView("flat")
flat = spark.table("flat")
print(len(flat.columns), "columns")

In [ ]:
FIELDS = {
    "incident_date":      resolve(flat, "incident_date", ["incident_Incident_Date", "Incident_Date", "Unit_Notified", "Dispatch_Date", "_Date"]),
    "incident_id":        resolve(flat, "incident_id", ["Incident_Transaction_GUID", "Incident_Number", "Response_Number", "PCR"]),
    "patient_id":         resolve(flat, "patient_id", ["patient_Patient_ID", "Patient_Key", "MRN", "Person_ID"]),
    "payer":              resolve(flat, "payer", ["payment_Primary_Method", "Payment_Method", "Payer", "Insurance_Company", "Billing_Type"]),
    "primary_impression": resolve(flat, "primary_impression", ["Primary_Impression", "Provider_Primary_Impression", "Impression"]),
    "acuity":             resolve(flat, "acuity", ["Acuity", "Severity", "Level_Of_Care", "Priority"]),
    "disposition":        resolve(flat, "disposition", ["Incident_Patient_Disposition", "Patient_Disposition", "disposition_Disposition"]),
    "state":              resolve(flat, "state", ["Scene_State", "Incident_State", "_State"]),
    "county":             resolve(flat, "county", ["Scene_County", "County"]),
    "zip":                resolve(flat, "zip", ["Scene_Zip", "Zip", "Postal"]),
    "agency_name":        resolve(flat, "agency_name", ["agency_Agency_Name", "Agency_Name", "agency_Name"]),
}

display(spark.createDataFrame([(k, v or "NOT FOUND") for k, v in FIELDS.items()], "field string, resolved_column string"))

In [ ]:
DATE = FIELDS["incident_date"]
base = (flat.withColumn("yr", F.year(F.col(DATE).cast("timestamp")))
             .withColumn("mo", F.date_format(F.col(DATE).cast("timestamp"), "yyyy-MM"))
             .withColumn("hr", F.hour(F.col(DATE).cast("timestamp")))
             .withColumn("dow", F.date_format(F.col(DATE).cast("timestamp"), "E")))
base.createOrReplaceTempView("base")
display(base.groupBy("yr").count().orderBy("yr"))

## 4. Volume

Look for partial years and agencies dropping in or out. Both throw off year-over-year numbers.

In [ ]:
scope = base.where((F.col("yr") >= YEAR_MIN) & (F.col("yr") <= YEAR_MAX))
scope.createOrReplaceTempView("scope")
display(scope.groupBy("mo").count().orderBy("mo"))

In [ ]:
display(scope.groupBy("yr", FIELDS["agency_name"]).count().orderBy("yr", F.desc("count")))

In [ ]:
if FIELDS["state"]:
    display(scope.groupBy(FIELDS["state"]).count().orderBy(F.desc("count")))

## 5. Completeness

Blank counts nulls and NEMSIS placeholders like "Not Recorded" and "Not Applicable". Earlier profiling found about 50% patient coverage but only about 10% of fields filled in, so the placeholders matter.

In [ ]:
KEY_FIELDS = [v for v in FIELDS.values() if v]
display(completeness(scope, KEY_FIELDS))

In [ ]:
CLINICAL = [c for c in scope.columns if any(k in c.lower() for k in
            ["impression", "symptom", "complaint", "vital", "medication", "procedure",
             "history", "disposition", "destination", "level_of_care", "cardiacarrest"])][:60]
display(completeness(scope, CLINICAL))

In [ ]:
n_by_yr = {r.yr: r["count"] for r in scope.groupBy("yr").count().collect()}
rows = []
for c in KEY_FIELDS:
    for r in scope.groupBy("yr").agg(F.count(F.when(~is_blank(c), 1)).alias("p")).collect():
        rows.append((c, r.yr, round(100.0 * r.p / n_by_yr[r.yr], 1)))
display(spark.createDataFrame(rows, "field string, yr int, pct_populated double").orderBy("field", "yr"))

## 6. Payer mix

Read the raw values before trusting the Medicaid flag. Managed care plan names often sit in the same column and will not match on the word medicaid.

In [ ]:
PAYER = FIELDS["payer"]
display(top_values(scope, PAYER, 50))

In [ ]:
flagged = scope.withColumn("payer_group",
    F.when(F.lower(F.col(PAYER)).rlike("medicaid|chip|title xix|managed care"), "Medicaid")
     .when(F.lower(F.col(PAYER)).rlike("medicare"), "Medicare")
     .when(F.lower(F.col(PAYER)).rlike("self|patient pay|uninsur|no insur"), "Self Pay")
     .when(is_blank(PAYER), "Unknown")
     .otherwise("Other/Commercial"))
flagged.createOrReplaceTempView("flagged")
display(flagged.groupBy("yr", "payer_group").count().orderBy("yr", F.desc("count")))

In [ ]:
if FIELDS["state"]:
    display(flagged.groupBy(FIELDS["state"]).pivot("payer_group").count())

## 7. How often patients come back

Groups patients into 1, 2-4, 5-11, and 12+ encounters. First check whether the patient ID stays the same across incidents. If incidents per patient comes back near 1.0, the ID is created fresh each call and none of this can be built from this table.

In [ ]:
PID = FIELDS["patient_id"]
med = flagged.where("payer_group = 'Medicaid'")
med.createOrReplaceTempView("med")
display(med.agg(F.count("*").alias("incidents"),
                F.countDistinct(PID).alias("distinct_patients"),
                F.round(F.count("*") / F.countDistinct(PID), 2).alias("incidents_per_patient")))

In [ ]:
per_pt = med.where(F.col(PID).isNotNull()).groupBy(PID).agg(F.count("*").alias("encounters"))
pyramid = (per_pt.withColumn("tier", F.when(F.col("encounters") == 1, "1")
                                      .when(F.col("encounters") <= 4, "2-4")
                                      .when(F.col("encounters") <= 11, "5-11")
                                      .otherwise("12+"))
                 .groupBy("tier").agg(F.count("*").alias("patients"), F.sum("encounters").alias("encounters")))
total_pt = per_pt.count()
total_enc = per_pt.agg(F.sum("encounters")).collect()[0][0]
display(pyramid.withColumn("pct_patients", F.round(100.0 * F.col("patients") / total_pt, 2))
               .withColumn("pct_encounters", F.round(100.0 * F.col("encounters") / total_enc, 2)))

In [ ]:
from pyspark.sql.window import Window

w = Window.partitionBy(PID).orderBy(F.col(DATE).cast("timestamp"))
gaps = (med.where(F.col(PID).isNotNull())
           .withColumn("next_dt", F.lead(F.col(DATE).cast("timestamp")).over(w))
           .withColumn("days_to_next", F.datediff("next_dt", F.col(DATE).cast("timestamp")))
           .where("days_to_next is not null"))
display(gaps.agg(F.count("*").alias("pairs"),
                 F.expr("percentile_approx(days_to_next, 0.5)").alias("median_days"),
                 F.round(100.0 * F.avg(F.when(F.col("days_to_next") <= 7, 1).otherwise(0)), 1).alias("pct_within_7d"),
                 F.round(100.0 * F.avg(F.when(F.col("days_to_next") <= 30, 1).otherwise(0)), 1).alias("pct_within_30d")))

## 8. Clinical and timing

In [ ]:
display(top_values(med, FIELDS["primary_impression"], 30))

In [ ]:
display(med.groupBy("dow", "hr").count().orderBy("dow", "hr"))

In [ ]:
if FIELDS["disposition"]:
    display(top_values(med, FIELDS["disposition"], 30))

In [ ]:
if FIELDS["acuity"]:
    display(flagged.groupBy(FIELDS["acuity"], "payer_group").count().orderBy(F.desc("count")))

In [ ]:
hist = [c for c in scope.columns if "history" in c.lower() or "comorb" in c.lower()]
print(hist)

## 9. ESO outcomes

Two things to find out: how many transported patients have an outcome record at all, and whether the ones that match are a skewed group. Hospitals were reportedly not sending low acuity records.

In [ ]:
eso = spark.table(ESO_TABLE)
print(eso.count(), len(eso.columns))
display(spark.createDataFrame([(c,) for c in eso.columns], "column string"))

In [ ]:
eso_keys = [c for c in eso.columns if any(k in c.lower() for k in ["incident", "response", "pcr", "run", "record", "guid", "event"])]
epcr_keys = [c for c in scope.columns if any(k in c.lower() for k in ["incident_number", "response_number", "transaction_guid", "pcr"])]
print(eso_keys)
print(epcr_keys)

In [ ]:
ESO_KEY = eso_keys[0] if eso_keys else None
EPCR_KEY = FIELDS["incident_id"]
display(overlap(scope, EPCR_KEY, eso, ESO_KEY))

In [ ]:
OUTCOME_COLS = [c for c in eso.columns if any(k in c.lower() for k in
                ["outcome", "diagnosis", "disposition", "discharge", "admit", "admission",
                 "mortality", "death", "icd", "ed_", "hospital"])][:40]
display(completeness(eso, OUTCOME_COLS))

In [ ]:
matched = (scope.join(eso.select(F.col(ESO_KEY).cast("string").alias("_k")).distinct(),
                      F.col(EPCR_KEY).cast("string") == F.col("_k"), "left")
                .withColumn("has_outcome", F.col("_k").isNotNull()))
matched.createOrReplaceTempView("matched")
display(matched.groupBy("yr").agg(F.count("*").alias("epcr_records"),
                                  F.sum(F.col("has_outcome").cast("int")).alias("with_outcome"),
                                  F.round(100.0 * F.avg(F.col("has_outcome").cast("int")), 1).alias("pct_matched")).orderBy("yr"))

In [ ]:
if FIELDS["acuity"]:
    display(matched.groupBy(FIELDS["acuity"]).agg(F.count("*").alias("n"),
            F.round(100.0 * F.avg(F.col("has_outcome").cast("int")), 1).alias("pct_matched")).orderBy(F.desc("n")))

In [ ]:
display(matched.groupBy(FIELDS["agency_name"]).agg(F.count("*").alias("n"),
        F.round(100.0 * F.avg(F.col("has_outcome").cast("int")), 1).alias("pct_matched")).orderBy(F.desc("n")).limit(40))

## 10. Geography

Shows the smallest level we can join public data (ACS, HPSA, CDC PLACES) to. County FIPS or ZIP is the realistic target.

In [ ]:
GEO = [v for v in [FIELDS["state"], FIELDS["county"], FIELDS["zip"]] if v]
display(completeness(med, GEO))

In [ ]:
if FIELDS["county"]:
    display(med.groupBy(FIELDS["state"], FIELDS["county"]).count().orderBy(F.desc("count")).limit(50))

## 11. What we can answer

Each question is scored by the weakest field it needs. Anything under 80% needs a fix or a caveat before it goes in front of leadership.

In [ ]:
QUESTIONS = {
    "Medicaid share of EMS volume": [PAYER, DATE],
    "Utilization pyramid (1 / 2-4 / 5-11 / 12+)": [PID, PAYER, DATE],
    "7- and 30-day recurrent demand": [PID, DATE],
    "Clinical mix / potentially avoidable episodes": [FIELDS["primary_impression"], PAYER],
    "Time-of-day and day-of-week demand": [DATE],
    "Transport vs non-transport disposition": [FIELDS["disposition"], PAYER],
    "Geographic overlay with ACS / HPSA / PLACES": [FIELDS["county"], FIELDS["zip"], FIELDS["state"]],
    "ED outcome linkage": [EPCR_KEY],
}

needed = list({f for v in QUESTIONS.values() for f in v if f})
pct = {r.field: r.pct_populated for r in completeness(med, needed).collect()}
rows = []
for q, fs in QUESTIONS.items():
    fs = [f for f in fs if f]
    worst = min([pct.get(f, 0.0) for f in fs]) if fs else 0.0
    verdict = "Ready" if worst >= 80 else ("Caveat needed" if worst >= 40 else "Blocked")
    rows.append((q, ", ".join(fs), worst, verdict))
display(spark.createDataFrame(rows, "question string, fields string, weakest_field_pct double, verdict string").orderBy(F.desc("weakest_field_pct")))

## 12. Open questions

- Does the patient ID stay the same across incidents or get created fresh each call? The first cell in section 7 answers it, and all the repeat use work depends on it.
- Do `silver_elite_dwgmr` and `silver_frn_qry_elite_dwgmr_repl` return the same row counts? Section 2 settles it.
- Does the payer field come from billing or from the crew at the scene? Changes how far the Medicaid flag can be trusted.
- Is the ESO match gap missing data or a bad join key? Confirm before treating it as missing.
- Image Trend (`silver_elite_dwamgh`) access, and whether it needs to be added in for national coverage.